<h1><span style="color:red">Examples of Spatial Statistics</span></h1>

Currently works with polygon data in SuAVE



## 1. Retrieve survey parameters from the URL

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        from IPython.display import HTML as _HTML
        display(_HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE parameters found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open this notebook via SuAVEDispatch from your survey in SuAVE.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: verify session loaded from Drive/cache --------------------------
from IPython.display import HTML
if su.ENV.colab:
    if _p is None:
        display(HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE session found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open SuAVEDispatch from the correct survey in SuAVE, then re-open this notebook.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
    display(HTML(
        '<div style="font-size:12px;padding:8px 12px;border-radius:4px;margin:3px 0;'
        'background:#e8f5e9;border:1px solid #81c784">'
        'Session loaded &mdash; survey <b>' + _p.get('survey', '?') + '</b>'
        ', user <b>' + _p.get('user', '?') + '</b>.'
        '<br><span style="color:#666;font-size:11px">'
        'Wrong survey? Go back to SuAVE, open the correct survey, and click Jupyter again.'
        '</span></div>'))
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    from IPython.display import HTML as _HTML
    display(_HTML(
        '<div style="background:#dc3545;color:white;padding:14px 16px;'
        'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
        '&#9888; Parameters not loaded.'
        '<br><span style="font-size:12px;font-weight:normal">'
        'Navigate to your survey in SuAVE and relaunch this notebook to load parameters.'
        '</span></div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()
# Variables used throughout this notebook
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

# NFS image paths (non-empty only when running on a hub with NFS storage)
localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images   # alias used by some notebooks

# Fetch and cache survey CSV for panellibs.extract_data()
absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)


In [ ]:
# common imports
from __future__ import print_function
import io
import json
import base64
import math

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
import branca.colormap as cm
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, Javascript

from shapely.geometry import Point
from libpysal.weights import Queen, KNN
from esda.moran import Moran, Moran_Local
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW

pd.set_option('display.max_colwidth', 0)

def printmd(string):
    display(Markdown(string))

# local imports
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint


## 2. Read the survey file and extract numeric variables

In [ ]:

# read the csv file
df = panellibs.extract_data(absolutePath + csv_file)# print(absolutePath + csv_file)

# create a list of variable names
variables_df = pd.DataFrame({'varname':df.columns})
printmd("<b><span style='color:red'>All variables in the survey file:</span></b>")

col = 0
for var in variables_df.varname.values:
    print(str(col) +":  "+ var)
    col = col+1


# create a dictionary of #number variables with abbreviated and full variable names 
var_list = {n[:n.index('#')]:n for n in variables_df.varname.values if '#number' in n}
printmd("<b><span style='color:red'>Numeric variables:</span></b>")

col = 0
for var in var_list:
    print(str(col) +":  "+ var)
    col = col+1

#create a dataframe of only #number variables
nums_df = df[[n for n in variables_df.varname.values if '#number' in n]]


## 2. Detect Geometry and Create GeoDataFrame

Supports WKT geometry columns (any column whose name contains `geometry`) and point data via `Latitude` / `Longitude` columns.

In [ ]:
df.columns = df.columns.str.strip()

geometry_col = next((c for c in df.columns if 'geometry' in c.lower()), None)
lat_col      = next((c for c in df.columns if 'latitude'  in c.lower()), None)
lon_col      = next((c for c in df.columns if 'longitude' in c.lower()), None)

if geometry_col:
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.GeoSeries.from_wkt(df[geometry_col]),
        crs='EPSG:4326'
    )
    print(f'Geometry source: WKT column "{geometry_col}"')
elif lat_col and lon_col:
    _df = df.dropna(subset=[lat_col, lon_col]).copy()
    _df['geometry'] = _df.apply(
        lambda r: Point(r[lon_col], r[lat_col]), axis=1
    )
    gdf = gpd.GeoDataFrame(_df, geometry='geometry', crs='EPSG:4326')
    print(f'Geometry source: lat={lat_col!r}, lon={lon_col!r}')
else:
    from IPython.display import HTML as _H
    display(_H('<div style="background:#dc3545;color:white;padding:12px;'
               'border-radius:6px;font-weight:bold">'
               '&#9888; No geometry detected. Ensure the survey has a WKT '
               'geometry column or Latitude/Longitude columns.</div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()

_geom_type = gdf.geometry.geom_type.iloc[0]
print(f'{len(gdf)} features loaded  |  geometry type: {_geom_type}')
print(f'Columns: {list(gdf.columns[:8])} ...')


## 3. Map of Features

In [ ]:
try:
    _centroid = gdf.geometry.centroid.unary_union.centroid
    _center   = [_centroid.y, _centroid.x]
except Exception:
    _center = [0, 0]

m = folium.Map(location=_center, zoom_start=4, tiles='CartoDB positron')

if _geom_type == 'Point':
    for _, row in gdf.iterrows():
        if not row.geometry.is_empty:
            _parts = [f'Lat: {row.geometry.y:.2f}, Lon: {row.geometry.x:.2f}']
            for _k in df.columns:
                if '#name' in _k.lower():
                    _parts.append(f'Name: {row[_k]}')
            _tip = '<br>'.join(_parts)
            folium.Marker(
                location=[row.geometry.y, row.geometry.x],
                icon=folium.Icon(color='blue', icon='info-sign'),
                tooltip=folium.Tooltip(_tip, sticky=True)
            ).add_to(m)
else:
    _obj_cols = gdf.select_dtypes(include='object').columns[:4].tolist()
    folium.GeoJson(
        json.loads(gdf.to_crs('EPSG:4326').to_json()),
        tooltip=folium.GeoJsonTooltip(fields=_obj_cols) if _obj_cols else None
    ).add_to(m)

display(m)


## 4. Select Variables for GWR

In [ ]:
numeric_cols = gdf.select_dtypes(include='number').columns.tolist()

dep_selector = widgets.Dropdown(
    options=numeric_cols,
    description='Dependent:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='460px'),
)
ind_selector = widgets.SelectMultiple(
    options=numeric_cols,
    value=numeric_cols[:min(2, len(numeric_cols))],
    description='Independent:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='460px', height='180px'),
    rows=8,
)
display(HTML('<b>Select dependent variable (y) and independent variables (X).</b>'
             '<br>Hold Ctrl/Cmd to multi-select independent variables.'))
display(dep_selector)
display(ind_selector)


In [ ]:
# Confirm selections — run after choosing variables above
dependent_var   = dep_selector.value
independent_vars = [v for v in list(ind_selector.value) if v != dependent_var]
if not independent_vars:
    raise ValueError('Select at least one independent variable different from the dependent.')
print(f'Dependent : {dependent_var}')
print(f'Independent ({len(independent_vars)}): {independent_vars}')


## 5. Run Geographically-Weighted Regression

For polygon data GWR uses Queen weights; for point data it uses 5-nearest-neighbors weights. Bandwidth is selected automatically via `Sel_BW`.

In [ ]:
# Prepare GWR dataset: keep only needed columns, drop rows with missing values
gwr_df = gdf[[dependent_var] + independent_vars + ['geometry']].copy()
n_before = len(gwr_df)
gwr_df = gwr_df.dropna().copy()
n_after = len(gwr_df)
print(f'Prepared {n_after} rows for GWR (dropped {n_before - n_after} with missing values).')

# Centroid coordinates
coords = list(zip(gwr_df.geometry.centroid.x, gwr_df.geometry.centroid.y))

# y and X — raw, unstandardized (matches app.py)
y = gwr_df[[dependent_var]].values
X = gwr_df[independent_vars].values

# Bandwidth selection
print('Selecting bandwidth...')
bw = Sel_BW(coords, y, X).search()
print(f'Bandwidth selected: {bw}')

# Fit GWR
gwr_results = GWR(coords, y, X, bw=bw).fit()
print(f'R\u00b2: {gwr_results.R2:.4f}')


In [ ]:
# Coefficient table
coeff_cols = ['Intercept'] + independent_vars
coeff_df   = pd.DataFrame(gwr_results.params, columns=coeff_cols)
coeff_df.index = gwr_df.index

# Store coefficients back with #number suffix
for col in coeff_df.columns:
    gwr_df[f'{col}#number'] = coeff_df[col]

# Residuals and fitted values
gwr_df['residual#number'] = gwr_results.resid_response.flatten()
gwr_df['fitted#number']   = gwr_results.predy.flatten()

print(f'R\u00b2: {gwr_results.R2:.4f}')
display(HTML('<b>GWR Coefficient Summary (first 5 rows):</b>'))
display(coeff_df.head())

# Downloads
_c_csv = coeff_df.to_csv(index=False)
_c_b64 = base64.b64encode(_c_csv.encode()).decode()
display(Javascript(f"""
var a = document.createElement('a');
a.download = 'gwr_coefficients.csv';
a.href = 'data:text/csv;base64,{_c_b64}';
document.body.appendChild(a); a.click(); document.body.removeChild(a);
"""))

_r_csv = gwr_df[['residual#number', 'fitted#number']].to_csv(index=False)
_r_b64 = base64.b64encode(_r_csv.encode()).decode()
display(Javascript(f"""
var a = document.createElement('a');
a.download = 'gwr_residuals.csv';
a.href = 'data:text/csv;base64,{_r_b64}';
document.body.appendChild(a); a.click(); document.body.removeChild(a);
"""))
print('Downloads triggered.')


## 6. Residuals Choropleth Map

In [ ]:
res_map = folium.Map(location=_center, zoom_start=4, tiles='CartoDB positron')

_res_min = gwr_df['residual#number'].min()
_res_max = gwr_df['residual#number'].max()

if _geom_type == 'Point':
    _cmap = cm.linear.RdYlBu_11.scale(_res_min, _res_max)
    for _, row in gwr_df.iterrows():
        _color = _cmap(row['residual#number'])
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=5, fill=True, fill_opacity=0.8,
            color=_color, fill_color=_color,
            tooltip=f"Residual: {row['residual#number']:.3f}"
        ).add_to(res_map)
    _cmap.caption = 'Residuals'
    _cmap.add_to(res_map)
else:
    folium.Choropleth(
        geo_data=json.loads(gwr_df.to_crs('EPSG:4326').to_json()),
        data=gwr_df.reset_index(),
        columns=['index', 'residual#number'],
        key_on='feature.id',
        fill_color='RdYlBu', fill_opacity=0.7, line_opacity=0.2,
        legend_name='Residuals'
    ).add_to(res_map)

display(res_map)


## 7. Spatial Autocorrelation of Residuals

Global and Local Moran's I computed on GWR residuals. Point data uses KNN (k=5); polygon data uses Queen contiguity.

In [ ]:
# Build spatial weights
if _geom_type == 'Point':
    w = KNN.from_dataframe(gwr_df, k=5)
else:
    w = Queen.from_dataframe(gwr_df)
w.transform = 'r'

# Global Moran's I
moran = Moran(gwr_df['residual#number'], w)
display(HTML(
    f'<b>Global Moran\u2019s I on residuals:</b> '
    f'{moran.I:.4f} &nbsp; (p = {moran.p_sim:.4f})'
))

# Local Moran's I
moran_loc = Moran_Local(gwr_df['residual#number'], w)
gwr_df['local_I#number'] = moran_loc.Is

# Local Moran's I map
local_map = folium.Map(location=_center, zoom_start=4, tiles='CartoDB positron')
_li_min = gwr_df['local_I#number'].min()
_li_max = gwr_df['local_I#number'].max()

if _geom_type == 'Point':
    _cmap2 = cm.linear.PuOr_11.scale(_li_min, _li_max)
    for _, row in gwr_df.iterrows():
        _color = _cmap2(row['local_I#number'])
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=5, fill=True, fill_opacity=0.8,
            color=_color, fill_color=_color,
            tooltip=f"Local I: {row['local_I#number']:.3f}"
        ).add_to(local_map)
    _cmap2.caption = "Local Moran's I"
    _cmap2.add_to(local_map)
else:
    folium.Choropleth(
        geo_data=json.loads(gwr_df.to_crs('EPSG:4326').to_json()),
        data=gwr_df.reset_index(),
        columns=['index', 'local_I#number'],
        key_on='feature.id',
        fill_color='PuOr', fill_opacity=0.7, line_opacity=0.2,
        legend_name="Local Moran's I"
    ).add_to(local_map)

display(local_map)


## 8. Save GWR Results to SuAVE (Optional)

The following variables derived from GWR can be added to a new survey:

- `residual#number` — difference between observed and predicted values
- `local_I#number` — Local Moran's I statistic (spatial autocorrelation)
- `coef_<var>#number` — local regression coefficient for each independent variable
- `fitted#number` — GWR predicted values

In [ ]:
# Build the merged dataframe: original df + selected GWR-derived columns
_gwr_new_cols = (
    [f'{c}#number' for c in coeff_cols]
    + ['residual#number', 'fitted#number', 'local_I#number']
)
_gwr_new_cols = [c for c in _gwr_new_cols if c in gwr_df.columns]

df_with_gwr = df.copy()
for _col in _gwr_new_cols:
    df_with_gwr[_col] = gwr_df[_col]

print('GWR-derived columns ready to save:')
for c in _gwr_new_cols:
    print(f'  {c}')


In [ ]:
# Save the new CSV and upload to SuAVE
new_file  = suaveint.save_csv_file(df_with_gwr, absolutePath, csv_file)
survey_name = input('Enter new survey name: ')
suaveint.create_survey(
    survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views
)
